In [5]:
#!pip install osmnx
#!pip install OSMPythonTools
#import sys
#!{sys.executable} -m pip install osmnx
# !{sys.executable} -m pip install folium

In [6]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import shapely
import numpy as np
import osmnx as ox
import folium
import mapclassify

parks = gpd.read_file("Parks_(2024).geojson")

In [7]:
parks.head()

,OBJECTID,NAME,OWNER,NOTES,CITY,FEATTYPE,STATUS,SOURCE,SOURCE_YEAR,ShapeSTLength,ShapeSTArea,geometry
0,1,Bob Jones Park,None,None,Southlake,Local Park,Existing,City of Southlake,2021,30037.587797,1.102229e+07,"POLYGON ((-97.13657 32.99702, -97.13731 32.997..."
1,2,Lonesome Dove Park,None,None,Southlake,Local Park,Existing,City of Southlake,2021,2760.737930,3.505679e+05,"POLYGON ((-97.12942 32.97459, -97.12871 32.974..."
2,3,Liberty Park at Sheltonwood,None,None,Southlake,Local Park,Existing,City of Southlake,2021,4120.174030,8.706581e+05,"POLYGON ((-97.14474 32.97113, -97.14425 32.971..."
3,4,Bicentennial Park,None,None,Southlake,Local Park,Existing,City of Southlake,2021,9201.496728,3.494158e+06,"POLYGON ((-97.15109 32.94557, -97.15233 32.945..."
4,5,Royal & Annie Smith Park,None,None,Southlake,Local Park,Existing,City of Southlake,2021,3494.508653,5.710544e+05,"POLYGON ((-97.19851 32.93797, -97.19708 32.937..."


In [8]:
place = "Dallas County, Texas, United States of America"
api = ox.geocoder.geocode_to_gdf(place)

In [ ]:
api.explore()

In [ ]:
#WAS TRYING TO GET STREET DATA FROM OSM
#kernel keeps dying?
# graph = ox.graph.graph_from_place(place)
# fig, ax = ox.plot.plot_graph(graph, figsize=(6, 6))

In [ ]:
#CODE TO UPLOAD DALLAS COUNTY SHAPE DATA (DIDN'T REALIZE ALREADY IN OSM)
#counties = gpd.read_file("Counties.geojson")

#counties.head()

#dallas_ct = counties[counties['COUNTY'] == 'Dallas']
#dallas_ct.head()

#dallas_ct.plot(figsize=(10, 10));

In [ ]:
api.head()

In [ ]:
#only grabbing the dallas geometry
dallas_geo = api['geometry'].iloc[0]

In [ ]:
#labelling all parks that have some area within Dallas County
parks["in_dallas"] = parks.intersects(dallas_geo)

In [ ]:
parks.head()

In [ ]:
#grabbing the subset of all parks that are in Dallas County
dal_parks = parks[parks['in_dallas'] == True]

In [ ]:
dal_parks.head()

In [ ]:
#grabbing all streets that are in Dallas County 
#this took 317 seconds to run (5.5 minutes) 

#we might want to consider excluding paths that are running/walking/pedestrian paths because these are the ones inside the parks typically 
# tags = {"highway": True}

In [ ]:
#grabbing only streets tagged as sidewalks (aka walking) 
tags_sidewalk = {
    "footway": "sidewalk",
    "sidewalk": True
}
streets_gdf = ox.features_from_place(place, tags_sidewalk)

In [ ]:
#projecting my gdfs to Texas coords
streets_gdf_projected = streets_gdf.to_crs("EPSG:3083")
dal_parks_projected = dal_parks.to_crs("EPSG:3083")

In [ ]:
#getting only the boundary of the parks and plotting
dal_parks_projected["boundary"] = dal_parks_projected.boundary
ax = dal_parks_projected.plot(color="white", edgecolor="black", figsize=(10, 10))

In [ ]:
dal_parks_projected.head()

In [ ]:
# .intersection() will use the boundaries geometry 
dal_parks_projected = dal_parks_projected.set_geometry('boundary')

points = dal_parks_projected.union_all().intersection(streets_gdf_projected.union_all())

# Switch geometry back to the full shape
dal_parks_projected = dal_parks_projected.set_geometry('geometry')

In [ ]:
#plot all parks
ax = dal_parks_projected.plot(figsize=(20, 20), color='g')
#plot the streets
streets_gdf_projected.plot(ax=ax, color = 'black')

#plot intersection points of parks and streets
for point in points.geoms:
    if point.geom_type == 'Point':
        # x = [point.x for point in points.geoms]
        # y = [point.y for point in points.geoms]
        ax.plot(*point.xy, color='blue', marker='o', markersize =5, label='Collection Points')
    else:
        print("not a point")
        continue

